In [13]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
================================================================================
NTO AI Final - Кейс «Потеряшки»
================================================================================

Задача: Восстановить потерянные позитивные взаимодействия пользователей
с книгами по неполным логам.

Метрика: NDCG@20

Пайплайн:
    1. Загрузка данных
    2. Feature Engineering
    3. Генерация кандидатов
    4. Ранжирование (CatBoost)
    5. Создание сабмита
================================================================================
"""

import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import warnings
import cupy as cp  # Замена numpy для GPU
import cudf        # Замена pandas для GPU
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# CatBoost импорты остаются те же
from catboost import CatBoostRanker, Pool
warnings.filterwarnings('ignore')

# =============================================================================
# ЯЧЕЙКА 1: Импорты и настройки
# =============================================================================

print("=" * 80)
print("NTO AI Final - Кейс «Потеряшки»")
print("=" * 80)

# Проверка CatBoost
try:
    from catboost import CatBoostRanker, Pool
    CATBOOST_AVAILABLE = True
    print("✓ CatBoost доступен")
except ImportError:
    CATBOOST_AVAILABLE = False
    print("✗ CatBoost не установлен, используется эвристический ранкер")

# =============================================================================
# ЯЧЕЙКА 2: Класс Dataset - Загрузка и хранение данных
# =============================================================================

class Dataset:
    """
    Класс для загрузки и хранения всех данных соревнования.
    """
    
    def __init__(self, data_dir):
        self.data_dir = Path(data_dir)
        self.interactions = None
        self.targets = None
        self.editions = None
        self.authors = None
        self.genres = None
        self.book_genres = None
        self.users = None
        
        self._load_all()
    
    def _load_all(self):
        """Загрузка всех CSV файлов"""
        print("\n[ЗАГРУЗКА ДАННЫХ]")
        print("-" * 40)
        
        # Основные файлы
        self.interactions = cudf.read_csv(self.data_dir / "interactions.csv")
        self.targets = cudf.read_csv(self.data_dir / "targets.csv")
        
        self.editions = cudf.read_csv(self.data_dir / "editions.csv")
        self.authors = cudf.read_csv(self.data_dir / "authors.csv")
        self.genres = cudf.read_csv(self.data_dir / "genres.csv")
        self.book_genres = cudf.read_csv(self.data_dir / "book_genres.csv")
        self.users = cudf.read_csv(self.data_dir / "users.csv")

        # Конвертация времени (cuDF поддерживает datetime)
        if "event_ts" in self.interactions.columns:
            self.interactions["event_ts"] = cudf.to_datetime(self.interactions["event_ts"])
        
        # Конвертация времени
        if "event_ts" in self.interactions.columns:
            self.interactions["event_ts"] = pd.to_datetime(self.interactions["event_ts"])
        
        print(f"  Взаимодействий: {len(self.interactions):,}")
        print(f"  Целевых пользователей: {len(self.targets):,}")
        print(f"  Изданий: {len(self.editions):,}")
        print(f"  Авторов: {len(self.authors):,}")
        print(f"  Жанров: {len(self.genres):,}")
        print(f"  Пользователей в справочнике: {len(self.users):,}")
    
    def show_info(self):
        """Показать информацию о данных"""
        print("\n[ИНФОРМАЦИЯ О ДАННЫХ]")
        print("=" * 60)
        
        # Взаимодействия
        print("\ninteractions.csv:")
        print(f"   Колонки: {list(self.interactions.columns)}")
        print(f"   Строк: {len(self.interactions):,}")
        print(f"   Уникальных пользователей: {self.interactions['user_id'].nunique():,}")
        print(f"   Уникальных изданий: {self.interactions['edition_id'].nunique():,}")
        print(f"   Типы событий: {self.interactions['event_type'].value_counts().to_dict()}")
        
        # Период данных
        min_ts = self.interactions["event_ts"].min()
        max_ts = self.interactions["event_ts"].max()
        print(f"   Период: {min_ts} - {max_ts}")
        
        # Целевые пользователи
        print("\ntargets.csv:")
        print(f"   Колонки: {list(self.targets.columns)}")
        print(f"   Целевых пользователей: {len(self.targets):,}")
        
        # Издания
        print("\neditions.csv:")
        print(f"   Колонки: {list(self.editions.columns)}")
        print(f"   Изданий: {len(self.editions):,}")
        print(f"   Уникальных книг: {self.editions['book_id'].nunique():,}")
        print(f"   Уникальных авторов: {self.editions['author_id'].nunique():,}")
        
        # Пример данных
        print("\n📋 Пример взаимодействий:")
        print(self.interactions.head(3).to_string())
        
        print("\n📋 Пример изданий:")
        print(self.editions.head(3).to_string())
    
    def get_positives(self):
        """Получить позитивные взаимодействия (wishlist=1, read=2)"""
        return self.interactions[self.interactions["event_type"].isin([1, 2])].copy()
    
    def get_seen_pairs(self):
        """Получить наблюдаемые позитивные пары (user_id, edition_id)"""
        positives = self.get_positives()
        return set(zip(positives["user_id"], positives["edition_id"]))

NTO AI Final - Кейс «Потеряшки»
✓ CatBoost доступен


In [14]:
# =============================================================================
# ЯЧЕЙКА 3: Feature Engineering
# =============================================================================

def build_features(dataset):
    """
    Построение признаков для генерации кандидатов и ранжирования.
    
    Возвращает словарь с различными признаками.
    """
    print("\n[ПОСТРОЕНИЕ ПРИЗНАКОВ]")
    print("-" * 40)
    
    features = {}
    positives = dataset.get_positives()
    
    # === Признаки изданий ===
    
    # Глобальная популярность издания
    features["edition_pop"] = (
        positives.groupby("edition_id")["user_id"]
        .nunique()
        .rename("edition_pop")
    )
    print(f"  edition_pop: {len(features['edition_pop']):,} записей")
    
    # Популярность по типам событий
    wishlist_pop = (
        positives[positives["event_type"] == 1]
        .groupby("edition_id")["user_id"]
        .nunique()
        .rename("wishlist_pop")
    )
    read_pop = (
        positives[positives["event_type"] == 2]
        .groupby("edition_id")["user_id"]
        .nunique()
        .rename("read_pop")
    )
    features["wishlist_pop"] = wishlist_pop
    features["read_pop"] = read_pop
    
    # Средний рейтинг издания
    ratings = positives[positives["rating"].notna()]
    if len(ratings) > 0:
        features["edition_rating"] = ratings.groupby("edition_id")["rating"].mean().rename("edition_rating")
        features["edition_rating_count"] = ratings.groupby("edition_id")["rating"].count().rename("edition_rating_count")
    
    # Недавняя популярность (последние 30 дней)
    max_ts = positives["event_ts"].max()
    cutoff = max_ts - pd.Timedelta(days=30)
    recent = positives[positives["event_ts"] >= cutoff]
    features["edition_recent_pop"] = (
        recent.groupby("edition_id")["user_id"]
        .nunique()
        .rename("edition_recent_pop")
    )
    print(f"  edition_recent_pop: {len(features['edition_recent_pop']):,} записей")
    
    # === Признаки пользователей ===
    
    # Активность пользователя
    features["user_activity"] = (
        positives.groupby("user_id")["edition_id"]
        .count()
        .rename("user_activity")
    )
    print(f"  user_activity: {len(features['user_activity']):,} записей")
    
    # Средний рейтинг пользователя
    user_ratings = positives[positives["rating"].notna()]
    if len(user_ratings) > 0:
        features["user_mean_rating"] = user_ratings.groupby("user_id")["rating"].mean().rename("user_mean_rating")
    
    # === Профили предпочтений ===
    
    # Профиль пользователя по жанрам
    user_genres = positives[["user_id", "edition_id"]].merge(
        dataset.editions[["edition_id", "book_id"]], on="edition_id", how="inner"
    ).merge(dataset.book_genres, on="book_id", how="inner")
    
    user_genre_profile = (
        user_genres.groupby(["user_id", "genre_id"])["edition_id"]
        .count()
        .reset_index()
        .rename(columns={"edition_id": "count"})
    )
    # Нормализация
    total = user_genre_profile.groupby("user_id")["count"].transform("sum")
    user_genre_profile["weight"] = user_genre_profile["count"] / total
    features["user_genre_profile"] = user_genre_profile
    print(f"  user_genre_profile: {len(user_genre_profile):,} записей")
    
    # Профиль пользователя по авторам
    user_authors = positives[["user_id", "edition_id"]].merge(
        dataset.editions[["edition_id", "author_id"]], on="edition_id", how="inner"
    )
    
    user_author_profile = (
        user_authors.groupby(["user_id", "author_id"])["edition_id"]
        .count()
        .reset_index()
        .rename(columns={"edition_id": "count"})
    )
    total = user_author_profile.groupby("user_id")["count"].transform("sum")
    user_author_profile["weight"] = user_author_profile["count"] / total
    features["user_author_profile"] = user_author_profile
    print(f"  user_author_profile: {len(user_author_profile):,} записей")
    
    # === Справочные таблицы ===
    
    # Жанры изданий
    edition_genres = dataset.editions[["edition_id", "book_id"]].merge(
        dataset.book_genres, on="book_id", how="left"
    )
    features["edition_genres"] = edition_genres
    
    # Авторы изданий
    features["edition_authors"] = dataset.editions[["edition_id", "author_id"]].copy()
    
    # Возрастные ограничения
    features["edition_age"] = dataset.editions.set_index("edition_id")["age_restriction"]
    
    # Год публикации
    features["edition_year"] = dataset.editions.set_index("edition_id")["publication_year"]
    
    print(f"\n  Всего создано {len(features)} групп признаков")
    return features

In [15]:
def generate_genre_candidates(dataset, features, user_ids, k=100):
    """
    Полностью векторизованная генерация кандидатов по жанрам на GPU (RAPIDS cuDF).
    """
    import cudf
    
    print("  [user_genre] Генерация по жанрам (GPU)...")

    # Получаем данные из features (ожидаем, что это cudf Series/DataFrame)
    user_genre_profile = features["user_genre_profile"]
    edition_pop = features["edition_pop"]       # Series: edition_id -> pop
    edition_genres = features["edition_genres"] # DataFrame: edition_id, book_id, genre_id

    # Выходим, если данных нет
    if user_genre_profile.empty or edition_genres.empty:
        return cudf.DataFrame(columns=["user_id", "edition_id", "score", "source"])

    # --- ШАГ 1: Ограничение словаря кандидатов (фильтрация изданий) ---
    # Чтобы не присоединять ВСЕ издания всех жанров, берем только топ популярных в каждом жанре.
    # Это аналогично блоку genre_to_editions в оригинале, но на GPU.
    
    top_per_genre = max(k * 5, 200)

    # 1.1. Присоединяем популярность к таблице жанров изданий
    # edition_pop -> DataFrame для merge
    pop_df = edition_pop.reset_index()
    pop_df.columns = ['edition_id', 'popularity']
    
    # Merge происходит на GPU мгновенно
    ed_genres_pop = edition_genres.merge(pop_df, on='edition_id', how='left')
    ed_genres_pop['popularity'] = ed_genres_pop['popularity'].fillna(0)

    # 1.2. Сортируем внутри каждого жанра по популярности
    ed_genres_pop = ed_genres_pop.sort_values(['genre_id', 'popularity', 'edition_id'], 
                                               ascending=[True, False, True])
    
    # 1.3. Оставляем только top_per_genre изданий в каждом жанре
    # Используем cumcount для ранжирования внутри группы на GPU
    ed_genres_pop['rank_in_genre'] = ed_genres_pop.groupby('genre_id').cumcount()
    top_editions_filtered = ed_genres_pop[ed_genres_pop['rank_in_genre'] < top_per_genre]
    
    # --- ШАГ 2: Соединение пользователей и кандидатов ---
    
    # Фильтруем профиль пользователей только по целевым user_ids
    # cudf.Series.isin работает очень быстро на GPU
    user_genre_profile = user_genre_profile[user_genre_profile['user_id'].isin(user_ids)]

    # Join: (user_id, genre_id, weight) + (edition_id, genre_id, popularity)
    # Результат: таблица, где каждому пользователю сопоставлены популярные книги его жанров
    candidates = user_genre_profile.merge(
        top_editions_filtered, 
        on='genre_id', 
        how='inner'
    )

    # --- ШАГ 3: Расчет скоров ---
    
    # Формула score: weight * (pop + 1)
    candidates['score'] = candidates['weight'] * (candidates['popularity'] + 1)

    # --- ШАГ 4: Агрегация и Топ-K ---
    
    # Одно издание могло попасться через разные жанры пользователя.
    # Группируем и суммируем score.
    candidates = candidates.groupby(['user_id', 'edition_id'], as_index=False)['score'].sum()

    # Ранжирование внутри пользователя
    # Сортируем: user (asc), score (desc), edition (asc - для стабильности)
    candidates = candidates.sort_values(['user_id', 'score', 'edition_id'], ascending=[True, False, True])

    # Берем топ-k через cumcount (аналог row_number в SQL)
    candidates['rank'] = candidates.groupby('user_id').cumcount()
    candidates = candidates[candidates['rank'] < k]
    
    # --- ШАГ 5: Финализация ---
    
    candidates = candidates.drop(columns=['rank'])
    candidates['source'] = 'user_genre'
    
    # Явное приведение типов для CatBoost/Pandas совместимости
    candidates['user_id'] = candidates['user_id'].astype('int64')
    candidates['edition_id'] = candidates['edition_id'].astype('int64')
    candidates['score'] = candidates['score'].astype('float32')

    print(f"    Сгенерировано: {len(candidates):,} кандидатов")
    
    # Возвращаем только нужные колонки
    return candidates[["user_id", "edition_id", "score", "source"]]

In [16]:
# =============================================================================
# ЯЧЕЙКА 5: Построение датасета для ранжирования
# =============================================================================

def build_ranking_dataset(dataset, features, candidates):
    """
    Построение датасета для ранжирования с признаками.
    """
    print("\n[ПОСТРОЕНИЕ ДАТАСЕТА ДЛЯ РАНЖИРОВАНИЯ]")
    print("-" * 40)

    seen_pairs = dataset.get_seen_pairs()

    # Фильтрация уже просмотренных
    candidates = candidates.copy()
    candidates["pair"] = list(zip(candidates["user_id"], candidates["edition_id"]))
    candidates = candidates[~candidates["pair"].isin(seen_pairs)]
    candidates = candidates.drop(columns=["pair"])

    # КРИТИЧЕСКИЙ ФИКС: убираем дубли user_id+edition_id,
    # сохраняя лучший score и все источники (multi-hot)
    pair_cols = ["user_id", "edition_id"]

    base = (
        candidates.groupby(pair_cols, as_index=False)["score"]
        .max()
    )

    if "source" in candidates.columns:
        src_flags = pd.get_dummies(candidates["source"], prefix="src")
        src_frame = pd.concat([candidates[pair_cols], src_flags], axis=1)
        src_frame = src_frame.groupby(pair_cols, as_index=False).max()
        df = base.merge(src_frame, on=pair_cols, how="left")
    else:
        df = base.copy()

    print(f"  Кандидатов после фильтрации и дедупликации: {len(df):,}")

    # Добавление признаков
    # Признаки изданий
    df = df.merge(features["edition_pop"].reset_index(), on="edition_id", how="left")
    df = df.merge(features["wishlist_pop"].reset_index(), on="edition_id", how="left")
    df = df.merge(features["read_pop"].reset_index(), on="edition_id", how="left")
    df = df.merge(features["edition_recent_pop"].reset_index(), on="edition_id", how="left")

    if "edition_rating" in features:
        df = df.merge(features["edition_rating"].reset_index(), on="edition_id", how="left")
    if "edition_rating_count" in features:
        df = df.merge(features["edition_rating_count"].reset_index(), on="edition_id", how="left")

    # Признаки пользователей
    df = df.merge(features["user_activity"].reset_index(), on="user_id", how="left")

    if "user_mean_rating" in features:
        df = df.merge(features["user_mean_rating"].reset_index(), on="user_id", how="left")

    # Возраст и год публикации
    edition_age = features["edition_age"].reset_index()
    edition_age.columns = ["edition_id", "age_restriction"]
    df = df.merge(edition_age, on="edition_id", how="left")

    edition_year = features["edition_year"].reset_index()
    edition_year.columns = ["edition_id", "publication_year"]
    df = df.merge(edition_year, on="edition_id", how="left")

    # Демография пользователей
    df = df.merge(dataset.users, on="user_id", how="left")

    # Заполнение пропусков
    df = df.fillna(0)

    print(f"  Размер датасета: {df.shape}")
    print(f"  Колонки: {list(df.columns)}")

    return df


In [17]:
# =============================================================================
# ЯЧЕЙКА 6: Ранжирование с CatBoost
# =============================================================================

# БЫСТРОЕ ИСПРАВЛЕНИЕ: переопределяем функцию с правильной loss-функцией

def train_catboost_ranker(train_df, feature_cols):
    """Обучение CatBoost Ranker на GPU"""
    print("\n[ОБУЧЕНИЕ CATBOOST RANKER НА GPU]")
    print("-" * 40)

    if not CATBOOST_AVAILABLE:
        print("  CatBoost недоступен")
        return None

    # Сортировка важна для групп
    train_df = train_df.sort_values("user_id")

    # Внутри train_catboost_ranker

    X_gpu = train_df[feature_cols].values # Вернет cupy.ndarray (массив на GPU)
    y_gpu = train_df["score"].values
    
    train_pool = Pool(
        data=X_gpu,
        label=y_gpu,
        group_id=train_df["user_id"].values
    )

    # ИЗМЕНЕНИЯ ЗДЕСЬ: task_type="GPU"
    model = CatBoostRanker(
        iterations=1000,           # Можно увеличить, GPU быстрее
        learning_rate=0.1,
        depth=6,
        loss_function="YetiLoss", # Хорошо работает на GPU для ранжирования (или QueryRMSE)
        task_type="GPU",          # <--- КЛЮЧЕВОЙ ПАРАМЕТР
        devices='0',              # Использовать GPU 0
        verbose=50,
        random_seed=42
    )

    model.fit(train_pool)
    print(f"  Модель обучена на GPU: {model.tree_count_} деревьев")

    return model


def rank_with_catboost(df, model, features_cols, k=20):
    """Ранжирование с помощью CatBoost"""
    print("\n[РАНЖИРОВАНИЕ CATBOOST]")
    print("-" * 40)
    
    if model is None:
        print("  Модель недоступна, используем эвристическое ранжирование")
        return rank_with_heuristics(df, k)
    
    # Предсказание
    X = df[features_cols].values
    df["pred"] = model.predict(X)
    
    # Сортировка и выбор топ-k
    df = df.sort_values(["user_id", "pred", "edition_id"], ascending=[True, False, True])
    
    result = df.groupby("user_id").head(k).copy()
    result["rank"] = result.groupby("user_id").cumcount() + 1
    
    result = result[["user_id", "edition_id", "rank", "pred"]].rename(columns={"pred": "score"})
    
    print(f"  Результат: {len(result):,} записей")
    
    return result


def rank_with_heuristics(df, k=20, source_weights=None):
    """Эвристическое ранжирование с весами источников"""
    print("\n[ЭВРИСТИЧЕСКОЕ РАНЖИРОВАНИЕ]")
    print("-" * 40)
    
    if source_weights is None:
        source_weights = {
            "src_user_genre": 1.0,
            "src_user_author": 1.0,
            "src_global_pop": 0.5,
            "src_item_knn": 0.8,
        }
    
    # Вычисление финального скора
    df["weight"] = 1.0
    for col, weight in source_weights.items():
        if col in df.columns:
            df.loc[df[col] == 1, "weight"] = weight
    
    df["final_score"] = df["score"] * df["weight"]
    
    # Сортировка
    df = df.sort_values(["user_id", "final_score", "edition_id"], ascending=[True, False, True])
    
    # Топ-k
    result = df.groupby("user_id").head(k).copy()
    result["rank"] = result.groupby("user_id").cumcount() + 1
    
    result = result[["user_id", "edition_id", "rank", "final_score"]].rename(columns={"final_score": "score"})
    
    print(f"  Результат: {len(result):,} записей")
    
    return result

In [18]:
# =============================================================================
# ЯЧЕЙКА 7: Валидация и создание сабмита
# =============================================================================

def validate_submission(submission, dataset, k=20):
    """Валидация сабмита"""
    print("\n[ВАЛИДАЦИЯ САБМИТА]")
    print("-" * 40)

    errors = []

    # Проверка колонок
    required = ["user_id", "edition_id", "rank"]
    missing = [c for c in required if c not in submission.columns]
    if missing:
        errors.append(f"Отсутствуют колонки: {missing}")

    # Проверка пользователей
    target_users = set(dataset.targets["user_id"])
    submitted_users = set(submission["user_id"])

    if target_users - submitted_users:
        errors.append(f"Отсутствуют пользователи: {len(target_users - submitted_users)}")
    if submitted_users - target_users:
        errors.append(f"Лишние пользователи: {len(submitted_users - target_users)}")

    # Проверка количества записей
    counts = submission.groupby("user_id").size()
    invalid = counts[counts != k]
    if len(invalid) > 0:
        errors.append(f"Неверное количество записей: {len(invalid)} пользователей")

    # Проверка рангов
    for user_id in list(target_users)[:10]:  # Проверка выборки
        user_ranks = sorted(submission[submission["user_id"] == user_id]["rank"].tolist())
        if user_ranks != list(range(1, k + 1)):
            errors.append(f"Неверные ранги для пользователя {user_id}")
            break

    # Проверка дубликатов
    dup = submission.groupby(["user_id", "edition_id"]).size()
    if (dup > 1).sum() > 0:
        errors.append(f"Дубликаты пар: {(dup > 1).sum()}")

    if errors:
        for err in errors:
            print(f"  ✗ {err}")
        raise ValueError("Валидация не пройдена!")
    else:
        print("  ✓ Валидация пройдена успешно!")


def make_submission(ranked, dataset, k=20):
    """Создание финального сабмита с заполнением пропусков"""
    print("\n[СОЗДАНИЕ САБМИТА]")
    print("-" * 40)

    submission = ranked[["user_id", "edition_id", "rank"]].copy()

    # КРИТИЧЕСКИЙ ФИКС: удаляем дубли user-edition и пересчитываем ранги
    if "score" in ranked.columns:
        tmp = ranked[["user_id", "edition_id", "score"]].copy()
        tmp = tmp.sort_values(["user_id", "score", "edition_id"], ascending=[True, False, True])
        tmp = tmp.drop_duplicates(subset=["user_id", "edition_id"], keep="first")
        tmp["rank"] = tmp.groupby("user_id").cumcount() + 1
        submission = tmp[tmp["rank"] <= k][["user_id", "edition_id", "rank"]]
    else:
        submission = submission.drop_duplicates(subset=["user_id", "edition_id"], keep="first")
        submission = submission.sort_values(["user_id", "rank", "edition_id"])
        submission["rank"] = submission.groupby("user_id").cumcount() + 1
        submission = submission[submission["rank"] <= k]

    # Фолбэк для всех пользователей с нехваткой рекомендаций
    positives = dataset.get_positives()
    pop = positives.groupby("edition_id")["user_id"].nunique().sort_values(ascending=False)
    top_editions = pop.head(k * 50).index.tolist()
    seen_pairs = dataset.get_seen_pairs()

    target_users = list(dataset.targets["user_id"])
    by_user = {uid: grp.copy() for uid, grp in submission.groupby("user_id")}

    rows = []
    for user_id in target_users:
        user_sub = by_user.get(user_id)
        used = set()

        if user_sub is not None and len(user_sub) > 0:
            user_sub = user_sub.sort_values("rank")
            for _, row in user_sub.iterrows():
                eid = int(row["edition_id"])
                if eid not in used:
                    rows.append({"user_id": int(user_id), "edition_id": eid, "rank": len(used) + 1})
                    used.add(eid)
                if len(used) == k:
                    break

        if len(used) < k:
            for edition_id in top_editions:
                if len(used) == k:
                    break
                if edition_id in used:
                    continue
                if (user_id, edition_id) in seen_pairs:
                    continue
                rows.append({
                    "user_id": int(user_id),
                    "edition_id": int(edition_id),
                    "rank": len(used) + 1
                })
                used.add(edition_id)

    submission = pd.DataFrame(rows, columns=["user_id", "edition_id", "rank"])

    # Финальная сортировка
    submission = submission.sort_values(["user_id", "rank"]).reset_index(drop=True)
    submission["user_id"] = submission["user_id"].astype("int64")
    submission["edition_id"] = submission["edition_id"].astype("int64")
    submission["rank"] = submission["rank"].astype("int64")

    print(f"  Размер сабмита: {len(submission):,} записей")
    print(f"  Уникальных пользователей: {submission['user_id'].nunique():,}")

    return submission


In [19]:
# =============================================================================
# ЯЧЕЙКА 8: Полный пайплайн
# =============================================================================

def run_pipeline(data_dir, output_path="submission.csv", k=20, candidates_per_user=100):
    """
    Запуск полного пайплайна.
    """
    print("\n" + "=" * 80)
    print("ЗАПУСК ПАЙПЛАЙНА")
    print("=" * 80)
    
    # 1. Загрузка данных
    dataset = Dataset(data_dir)
    dataset.show_info()
    
    # 2. Построение признаков
    features = build_features(dataset)
    
    # 3. Генерация кандидатов
    user_ids = dataset.targets["user_id"].values
    candidates = generate_all_candidates(dataset, features, user_ids, k=candidates_per_user)
    
    # 4. Построение датасета для ранжирования
    ranking_df = build_ranking_dataset(dataset, features, candidates)
    
    # 5. Определение признаков для модели
    feature_cols = [
        "score",
        "edition_pop", "wishlist_pop", "read_pop", "edition_recent_pop",
        "user_activity", "age_restriction", "publication_year",
        "gender", "age"
    ]
    # Добавляем one-hot источники
    src_cols = [c for c in ranking_df.columns if c.startswith("src_")]
    feature_cols.extend(src_cols)
    
    # Фильтруем существующие колонки
    feature_cols = [c for c in feature_cols if c in ranking_df.columns]
    print(f"\n  Признаков для модели: {len(feature_cols)}")
    
    # 6. Обучение и ранжирование
    if CATBOOST_AVAILABLE:
        model = train_catboost_ranker(ranking_df, feature_cols)
        ranked = rank_with_catboost(ranking_df, model, feature_cols, k)
    else:
        ranked = rank_with_heuristics(ranking_df, k)
    
    # 7. Создание сабмита
    submission = make_submission(ranked, dataset, k)
    
    # 8. Валидация
    validate_submission(submission, dataset, k)
    
    # 9. Сохранение
    submission.to_csv(output_path, index=False)
    print(f"\n✓ Сабмит сохранён: {output_path}")
    
    print("\n" + "=" * 80)
    print("ПАЙПЛАЙН ЗАВЕРШЁН")
    print("=" * 80)
    
    return submission

In [20]:
# =============================================================================
# ЯЧЕЙКА 9: Запуск в Jupyter/Kaggle
# =============================================================================

# Укажите путь к данным здесь:
DATA_DIR = "/kaggle/input/datasets/artemnazemtsev/nto-ai"  # <-- ИЗМЕНИТЕ ПУТЬ!
OUTPUT_PATH = "submission.csv"

# Проверка пути
import os
if os.path.exists(DATA_DIR):
    print(f"✓ Данные найдены: {DATA_DIR}")
    print(f"  Файлы: {os.listdir(DATA_DIR)}")
else:
    print(f"✗ Путь не найден: {DATA_DIR}")
    print("  Укажите правильный путь к данным в переменной DATA_DIR")

✓ Данные найдены: /kaggle/input/datasets/artemnazemtsev/nto-ai
  Файлы: ['book_genres.csv', 'users.csv', 'editions.csv', 'genres.csv', 'targets.csv', 'interactions.csv', 'authors.csv']


In [21]:
# Запуск
if os.path.exists(DATA_DIR):
    submission = run_pipeline(
        data_dir=DATA_DIR,
        output_path=OUTPUT_PATH,
        k=20,
        candidates_per_user=100
    )
    
    print("\n" + "=" * 60)
    print("ИТОГИ")
    print("=" * 60)
    print(f"Всего записей: {len(submission):,}")
    print(f"Уникальных пользователей: {submission['user_id'].nunique():,}")
    print(f"Уникальных изданий: {submission['edition_id'].nunique():,}")
    print("\nПример сабмита:")
    print(submission.head(25).to_string())
else:
    print("Сначала укажите правильный путь к данным!")


ЗАПУСК ПАЙПЛАЙНА

[ЗАГРУЗКА ДАННЫХ]
----------------------------------------
  Взаимодействий: 443,278
  Целевых пользователей: 3,862
  Изданий: 460,599
  Авторов: 100,724
  Жанров: 707
  Пользователей в справочнике: 80,738

[ИНФОРМАЦИЯ О ДАННЫХ]

interactions.csv:
   Колонки: ['user_id', 'edition_id', 'event_type', 'rating', 'event_ts']
   Строк: 443,278
   Уникальных пользователей: 19,259
   Уникальных изданий: 126,002
   Типы событий: {2: 258529, 1: 184749}
   Период: 2025-05-01 00:00:47 - 2025-11-30 23:59:35

targets.csv:
   Колонки: ['user_id']
   Целевых пользователей: 3,862

editions.csv:
   Колонки: ['edition_id', 'book_id', 'author_id', 'publication_year', 'age_restriction', 'language_id', 'publisher_id', 'title', 'description']
   Изданий: 460,599
   Уникальных книг: 337,245
   Уникальных авторов: 100,724

📋 Пример взаимодействий:
    user_id  edition_id  event_type  rating            event_ts
0  10000240  1000450846           1     NaN 2025-07-12 15:10:47
1  10000240  10005

    Сгенерировано: 385,700 кандидатов
  [user_author] Генерация по авторам...


    Сгенерировано: 379,081 кандидатов
  [global_pop] Генерация по популярности...


    Сгенерировано: 386,200 кандидатов
  [item_knn] Генерация по похожим изданиям...


    Сгенерировано: 378,526 кандидатов

  Всего кандидатов: 1,529,507
  Уникальных пар (user, edition): 1,015,867

[ПОСТРОЕНИЕ ДАТАСЕТА ДЛЯ РАНЖИРОВАНИЯ]
----------------------------------------
  Кандидатов после фильтрации: 1,430,958
  Размер датасета: (1430958, 19)
  Колонки: ['user_id', 'edition_id', 'score', 'edition_pop', 'wishlist_pop', 'read_pop', 'edition_recent_pop', 'edition_rating', 'edition_rating_count', 'user_activity', 'user_mean_rating', 'age_restriction', 'publication_year', 'gender', 'age', 'src_global_pop', 'src_item_knn', 'src_user_author', 'src_user_genre']

  Признаков для модели: 14

[ОБУЧЕНИЕ CATBOOST RANKER]
----------------------------------------
0:	learn: 86.9366507	total: 258ms	remaining: 2m 8s
50:	learn: 42.2125545	total: 7.32s	remaining: 1m 4s
100:	learn: 37.7332719	total: 14.1s	remaining: 55.7s
150:	learn: 36.0262468	total: 20.7s	remaining: 47.9s
200:	learn: 34.8615053	total: 27.4s	remaining: 40.8s
250:	learn: 33.9193652	total: 34.2s	remaining: 33.9s
300

ValueError: Валидация не пройдена!